# Tutorial - Seals

There are three physics-based seal models available in ROSS:

- `LabyrinthSeal` — multi-tooth compressible throttling seal.

- `HolePatternSeal` — bulk-flow model for annular seals with hole / honeycomb–type cells (pocket damper seals).

- `HybridSeal` — combined hole-pattern + labyrinth with iterative **_mass-flow matching_** at the interface.

When stiffness and damping coefficients are already known use the general class `SealElement` instead. It does not compute coefficients from geometry — you supply them directly.

>**Note**: Seal elements are excluded from static analysis as they do not support the rotor weight.

In [1]:
# Make sure the default renderer is set to 'notebook_connected' for inline plots in Jupyter
import plotly.io as pio

import ross  # importing ross sets its own default renderer

pio.renderers.default = "notebook_connected"

/home/raphaelts/ross-wt-release-3.2.0/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Section 1: Labyrinth Seal

A labyrinth restricts leakage through a series of throttlings (teeth). ROSS solves compressible flow along the seal, estimates **mass flow** (including choke where relevant), **swirl** in cavities, and **perturbed** equations for **stiffness and damping** (and optional fluid-related mass terms) versus whirl speed.

**Typical inputs:** shaft diameter, radial clearance, number of teeth, pitch and tooth geometry, seal type (`"rotor"`, `"stator"`, or `"inter"`), pressures, temperature, shaft speed(s), preswirl, and gas data.

In [2]:
from ross.seals.labyrinth_seal import LabyrinthSeal
from ross.units import Q_

seal = LabyrinthSeal(
    n=0,
    shaft_diameter=Q_(145, "mm"),
    radial_clearance=Q_(0.3, "mm"),
    n_teeth=16,
    pitch=Q_(3.175, "mm"),
    tooth_height=Q_(3.175, "mm"),
    tooth_width=Q_(0.1524, "mm"),
    seal_type="inter",
    inlet_pressure=308_000.0,
    outlet_pressure=94_300.0,
    inlet_temperature=283.15,
    speed=Q_([5000, 8000, 11000], "RPM"),
    preswirl=0.98,
    gas_composition={"Nitrogen": 0.79, "Oxygen": 0.21},
)
seal

LabyrinthSeal(n=0, n_link=None,
 kxx=[-19444.950751632474, -50204.727902921775, -100983.52983738085], kxy=[16547.934075144054, 35381.870773149814, 66099.01138927437],
 kyx=[-16547.934075144054, -35381.870773149814, -66099.01138927437], kyy=[-19444.950751632474, -50204.727902921775, -100983.52983738085],
 kzz=[0.0, 0.0, 0.0], cxx=[14.695413606002612, 23.68925805802294, 38.632064690433175],
 cxy=[35.425201448109505, 55.97903393756379, 80.95189704777692], cyx=[-35.425201448109505, -55.97903393756379, -80.95189704777692],
 cyy=[14.695413606002612, 23.68925805802294, 38.632064690433175], czz=[0.0, 0.0, 0.0],
 mxx=[0.0, 0.0, 0.0], mxy=[0.0, 0.0, 0.0],
 myx=[0.0, 0.0, 0.0], myy=[0.0, 0.0, 0.0],
 mzz=[0.0, 0.0, 0.0],
 speed=[ 523.5987756   837.75804096 1151.91730632], frequency=None, tag=None)

**Access:** Coefficients are stored as lists aligned with `speed` (for example `seal.kxx`, `seal.kxy`, …; `seal.cxx`, …). Leakage: `seal.seal_leakage` — total mass flow rate in **kg/s**, one value per entry in `speed`.

**Without** `gas_composition`, you must supply **`molar_mass`**, **`gamma`**, **`reference_temperatures`** (two temperatures [K]), and **`reference_viscosities`** (two dynamic viscosities [Pa·s]).

# Section 2: Hole-pattern Seal

Hole-pattern and honeycomb seals are represented by a one-dimensional **bulk-flow** model of the annular clearance: the base flow is solved along the seal, and a perturbation of it gives the reaction force to a whirling rotor. `kxx`, `kxy`, `cxx` and `cxy` are the coefficients at the whirl frequency `excitation_ratio * speed`, and the mass terms are zero; pass `frequency=` for a table over whirl frequencies (Section 4).

`gas_model="ideal"` (default) uses a perfect gas with the Sutherland viscosity, and `gas_model="real"` tabulates the gas with the equation of state of `ccp` (it needs `gas_composition`). The cells enter only through `cell_depth`, which for a hole pattern is the hole depth times the hole-area ratio; `cell_length` and `cell_width` are not used. The equations and their assumptions are on the [hole-pattern theory page](../theory_and_code/holepattern_seal.ipynb).

In [3]:
from ross.seals.holepattern_seal import HolePatternSeal
from ross.units import Q_

DRUM_OD_M = 196.5e-3
shaft_diameter = DRUM_OD_M
length_honeycomb = 84.9e-3
diametral_clearance = 0.36e-3
radial_clearance = diametral_clearance / 2

holepattern = HolePatternSeal(
    n=0,
    shaft_diameter=shaft_diameter,
    radial_clearance=radial_clearance,
    axial_length=length_honeycomb,
    relative_roughness=1.0e-4,
    cell_length=length_honeycomb / 37,
    cell_width=2.1e-3,
    cell_depth=2.8e-3,
    inlet_pressure=1_830_000.0,
    outlet_pressure=823_500.0,
    inlet_temperature=300.0,
    speed=Q_([5000], "RPM"),
    gas_composition={
        "Nitrogen": 0.7812,
        "Oxygen": 0.2096,
        "Argon": 0.0092,
    },
    preswirl=1.0,
    entrance_loss_coefficient=0.1,
    exit_loss_coefficient=0.5,
    nz=40,
)
holepattern

HolePatternSeal(n=0, n_link=None,
 kxx=[7735779.644606069], kxy=[8023113.307245199],
 kyx=[-8023113.307245199], kyy=[7735779.644606069],
 kzz=[0.0], cxx=[17116.02096709705],
 cxy=[-10218.499866087917], cyx=[10218.499866087917],
 cyy=[17116.02096709705], czz=[0.0],
 mxx=[0.0], mxy=[0.0],
 myx=[0.0], myy=[0.0],
 mzz=[0.0],
 speed=[523.5987756], frequency=None, tag=None)

# Section 3: Hybrid Seal

This tutorial demonstrates how to use the `HybridSeal` class to model a hybrid seal that combines a hole-pattern seal (damping section) with a labyrinth seal (throttling section). The hybrid seal model iteratively determines the interface pressure between the two seal stages by matching their leakage rates, then combines their rotordynamic coefficients.

The hybrid seal configuration in ROSS consists of:

1. **Hole-Pattern Seal (Damping Stage)**: Located upstream, this section is meant to add damping for rotor stability. In the model the cells act only as gas storage (see the theory page of `HolePatternSeal`).

2. **Labyrinth Seal (Throttling Stage)**: Located downstream, this section provides the primary pressure drop through a series of throttling cavities (teeth). The labyrinth geometry creates a tortuous flow path that minimizes leakage.

## 3.1 Iterative Pressure Matching

The model uses a bisection method to find the interface pressure ($P_{int}$) between the two seal stages that satisfies mass conservation:

$$\dot{m}_{hole-pattern}(P_{in}, P_{int}) = \dot{m}_{labyrinth}(P_{int}, P_{out})$$

where:
- $P_{in}$ is the inlet pressure (upstream of hole-pattern seal)
- $P_{out}$ is the outlet pressure (downstream of labyrinth seal)
- $\dot{m}$ represents the mass flow rate (leakage)

## 3.2 Hybrid Seal Parameters

The `HybridSeal` requires parameters for both the hole-pattern and labyrinth seal sections, as well as common operating conditions.

### 3.2.1 Common Parameters

These parameters are shared between both seal stages:

| Parameter | Description | Unit |
|-----------|-------------|------|
| `n` | Node location in the rotor model | - |
| `shaft_diameter` | Diameter of the shaft | m |
| `inlet_pressure` | Total inlet pressure at hole-pattern entrance | Pa |
| `outlet_pressure` | Final outlet pressure at labyrinth exit | Pa |
| `inlet_temperature` | Inlet temperature | K |
| `speed` | Rotor speed(s) | rad/s |
| `frequency` | Whirl frequencies of a 2-D table (optional) | rad/s |
| `gas_composition` | Gas composition dictionary (optional) | - |
| `molar_mass` | Molecular mass (required if no gas_composition) | kg/kgmol |
| `gamma` | Specific heat ratio Cp/Cv (required if no gas_composition) | - |

In [4]:
from ross import HybridSeal
from ross.units import Q_

# Common parameters for the hybrid seal
n = 3  # Node location
shaft_diameter = Q_(50, "mm")  # 50 mm shaft diameter
inlet_pressure = 500000  # 5 bar inlet pressure (Pa)
outlet_pressure = 100000  # 1 bar outlet pressure (Pa)
inlet_temperature = 300.0  # 300 K (~27°C)
speed = Q_([2000, 3000, 4000, 5000], "RPM")  # Operating speeds

# Gas composition for air
gas_composition = {
    "Nitrogen": 0.7812,
    "Oxygen": 0.2096,
    "Argon": 0.0092,
}

### 3.2.2 Hole-Pattern Seal Parameters

The hole-pattern seal (damping stage) parameters define the geometry of the porous surface:

| Parameter | Description | Unit |
|-----------|-------------|------|
| `radial_clearance` | Seal clearance | m |
| `axial_length` | Length of the seal | m |
| `relative_roughness` | Rotor surface roughness divided by the radial clearance | - |
| `cell_length` | Axial length of each cell (not used by the model) | m |
| `cell_width` | Circumferential width of each cell (not used by the model) | m |
| `cell_depth` | Depth of the cells (hole pattern: hole depth times hole-area ratio) | m |
| `preswirl` | Inlet swirl ratio | - |
| `entrance_loss_coefficient` | Entrance loss coefficient | - |
| `exit_loss_coefficient` | Exit loss coefficient | - |
| `gas_model` | `"ideal"` (default) or `"real"`; `"real"` requires `gas_composition` | - |

In [5]:
hole_pattern_params = {
    "radial_clearance": 0.0003,  # 0.3 mm clearance
    "axial_length": 0.04,  # 40 mm length
    "relative_roughness": 0.0001,  # Relative roughness
    "cell_length": 0.003,  # 3 mm cell length
    "cell_width": 0.003,  # 3 mm cell width
    "cell_depth": 0.002,  # 2 mm cell depth
    "preswirl": 0.8,  # 80% preswirl
    "entrance_loss_coefficient": 0.5,  # Entrance loss coefficient
    "exit_loss_coefficient": 1.0,  # Exit loss coefficient
}

### 3.2.3 Labyrinth Seal Parameters

The labyrinth seal (throttling stage) parameters define the tooth geometry:

| Parameter | Description | Unit |
|-----------|-------------|------|
| `radial_clearance` | Nominal radial clearance | m |
| `n_teeth` | Number of teeth (throttlings) | - |
| `pitch` | Axial cavity length (land length) | m |
| `tooth_height` | Height of seal strip | m |
| `tooth_width` | Thickness of throttle (tip-width) | m |
| `seal_type` | Location of teeth: 'rotor', 'stator', or 'inter' | - |
| `preswirl` | Inlet swirl velocity ratio | - |
| `reference_temperatures` | Temperature at states [T1, T2] (required if no gas_composition) | K |
| `reference_viscosities` | Dynamic viscosity at states [μ1, μ2] (required if no gas_composition) | kg/(m·s) |


In [6]:
labyrinth_params = {
    "radial_clearance": Q_(0.25, "mm"),  # 0.25 mm clearance
    "n_teeth": 10,  # 10 teeth
    "pitch": Q_(3, "mm"),  # 3 mm pitch
    "tooth_height": Q_(3, "mm"),  # 3 mm tooth height
    "tooth_width": Q_(0.15, "mm"),  # 0.15 mm tooth width
    "seal_type": "inter",  # Interlocking type
    "preswirl": 0.9,  # 90% preswirl
    "reference_temperatures": [300.0, 299.5],  # Temperature at states (K)
    "reference_viscosities": [1.85e-05, 1.84e-05],  # Dynamic viscosity (kg/(m·s))
}

## 3.3 Creating the Hybrid Seal

Now we can create the `HybridSeal` object:

In [7]:
hybrid_seal = HybridSeal(
    n=n,
    shaft_diameter=shaft_diameter,
    inlet_pressure=inlet_pressure,
    outlet_pressure=outlet_pressure,
    inlet_temperature=inlet_temperature,
    speed=speed,
    gas_composition=gas_composition,
    hole_pattern_parameters=hole_pattern_params,
    labyrinth_parameters=labyrinth_params,
)

## 3.4 Analyzing the Results

### 3.4.1 Summary Results

The `summary_results()` method provides a quick overview of the hybrid seal analysis:


In [8]:
hybrid_seal.summary_results()

Parameter,Value
Number of Iterations,21
Final Convergence,4.441881e-07
Interface Pressure (Pa),473863.03
Seal Leakage (kg/s),1.289588e-02


### 3.4.2 Rotordynamic Coefficients

The `format_table()` method displays the rotordynamic coefficients at each speed (whirl frequency = `excitation_ratio * speed`):


In [9]:
hybrid_seal.format_table(frequency_units="RPM")

Speed [RPM],kxx [N/m],kyy [N/m],kxy [N/m],kyx [N/m],kzz [N/m],cxx [N*s/m],cyy [N*s/m],cxy [N*s/m],cyx [N*s/m],czz [N*s/m]
2000.0,-469.85816,-469.85816,7793.99868,-7793.99868,0.0,46.03225,46.03225,3.04358,-3.04358,0.0
2750.0,-1037.54312,-1037.54312,10760.76081,-10760.76081,0.0,46.13054,46.13054,4.17828,-4.17828,0.0
3500.0,-1774.18275,-1774.18275,13760.23151,-13760.23151,0.0,46.25415,46.25415,5.31085,-5.31085,0.0
4250.0,-2690.42023,-2690.42023,16797.21943,-16797.21943,0.0,46.40492,46.40492,6.44169,-6.44169,0.0
5000.0,-3788.3248,-3788.3248,19876.67917,-19876.67917,0.0,46.58383,46.58383,7.57115,-7.57115,0.0


### 3.4.3 Convergence Analysis

The `plot_convergence()` method shows how the iterative solution converges to the final interface pressure. The convergence plot shows:
1. **Convergence History**: The relative leakage error decreasing with each iteration

2. **Leakage Rate Convergence**: How the leakage rates from both seals approach each other

3. **Interface Pressure Evolution**: The bisection method narrowing down to the correct interface pressure


In [10]:
fig_conv = hybrid_seal.plot_convergence()
fig_conv.show()

### 3.4.4 Pressure Distribution

The `plot_pressure_distribution()` method shows the pressure profile through both seal stages:


In [11]:
fig_pressure = hybrid_seal.plot_pressure_distribution(pressure_units="bar")
fig_pressure.show()

## 3.5 Alternative Configuration


If CoolProp is not available or you prefer to specify gas properties manually, you can omit the `gas_composition` parameter and provide `molar_mass` and `gamma` instead. In this case, you also need to provide viscosity model parameters for the hole-pattern seal (`sutherland_b`, `sutherland_s`).

In [12]:
# Hole-pattern parameters with manual viscosity model
hole_pattern_params_manual = {
    "radial_clearance": 0.0003,
    "axial_length": 0.04,
    "relative_roughness": 0.0001,
    "cell_length": 0.003,
    "cell_width": 0.003,
    "cell_depth": 0.002,
    "preswirl": 0.8,
    "entrance_loss_coefficient": 0.5,
    "exit_loss_coefficient": 1.0,
    "sutherland_b": 1.458e-6,  # Sutherland viscosity coefficient b
    "sutherland_s": 110.4,  # Sutherland viscosity coefficient s
}

# Create hybrid seal with manual gas properties
hybrid_seal_manual = HybridSeal(
    n=3,
    shaft_diameter=Q_(50, "mm"),
    inlet_pressure=500000,
    outlet_pressure=100000,
    inlet_temperature=300.0,
    speed=Q_([3000], "RPM"),
    molar_mass=28.97,  # Air molecular mass (kg/kgmol)
    gamma=1.4,  # Air specific heat ratio
    hole_pattern_parameters=hole_pattern_params_manual,
    labyrinth_parameters=labyrinth_params,
)

hybrid_seal_manual.summary_results()

Parameter,Value
Number of Iterations,23
Final Convergence,2.011831e-08
Interface Pressure (Pa),473944.71
Seal Leakage (kg/s),1.288200e-02


# Section 4: Frequency-dependent (asynchronous) coefficients

The tables built so far are **synchronous**: each entry of `speed` is solved with the whirl frequency equal to the rotor speed, which is the right assumption for the response to unbalance. Gas-seal forces, however, depend on two different frequencies:

- the **rotor speed** sets the base flow — leakage, cavity pressures and the swirl that feeds the cross-coupled stiffness;
- the **whirl (excitation) frequency** of the orbit sets the unsteady reaction — the damping, and the frequency dependence of the stiffness.

For a subsynchronous instability the destabilizing mode whirls close to the first natural frequency, well below the running speed, so its coefficients should be evaluated *at that whirl frequency*. Passing `frequency=` to a seal builds a 2-D table over `(speed, frequency)`; the base flow is solved once per speed and the perturbation once per whirl frequency.

In [13]:
from ross.seals.labyrinth_seal import LabyrinthSeal
from ross.units import Q_
import numpy as np

speed = Q_([5000, 8000, 11000], "RPM").to("rad/s").m
whirl = (
    Q_([2000, 4000, 8000, 12000], "RPM").to("rad/s").m
)  # must be strictly increasing

laby_params = dict(
    n=3,
    shaft_diameter=Q_(145, "mm"),
    radial_clearance=Q_(0.3, "mm"),
    n_teeth=16,
    pitch=Q_(3.175, "mm"),
    tooth_height=Q_(3.175, "mm"),
    tooth_width=Q_(0.1524, "mm"),
    seal_type="inter",
    inlet_pressure=308_000.0,
    outlet_pressure=94_300.0,
    inlet_temperature=283.15,
    preswirl=0.98,
    gas_composition={"Nitrogen": 0.79, "Oxygen": 0.21},
)

seal_sync = LabyrinthSeal(speed=speed, **laby_params)
seal_2d = LabyrinthSeal(speed=speed, frequency=whirl, **laby_params)

print(
    "1-D table kind:",
    seal_sync.kxx_interpolated.kind,
    "| shape:",
    np.array(seal_sync.kxx).shape,
)
print(
    "2-D table kind:",
    seal_2d.kxx_interpolated.kind,
    "| shape:",
    np.array(seal_2d.kxx).shape,
)

1-D table kind: speed | shape: (3,)
2-D table kind: grid | shape: (3, 4)


`seal_2d.kxx` is now a nested list with one row per speed and one column per whirl frequency. The point where the whirl frequency equals the speed reproduces the synchronous table exactly, and the leakage (a base-flow quantity) is still one value per speed:

In [14]:
kxy_2d = np.array(seal_2d.kxy)

# whirl[2] == speed[1] == 8000 RPM: the synchronous diagonal of the grid
print("kxy at 8000 RPM, whirl 8000 RPM :", kxy_2d[1, 2])
print("kxy from the synchronous table  :", seal_sync.kxy[1])
print("kxy at 8000 RPM, whirl 2000 RPM :", kxy_2d[1, 0])
print("leakage per speed [kg/s]        :", np.round(seal_2d.seal_leakage, 5))

seal_2d.format_table(
    speed=speed[1],
    frequency=whirl,
    coefficients=["kxx", "kxy", "cxx", "cxy"],
    frequency_units="RPM",
)

kxy at 8000 RPM, whirl 8000 RPM : 35381.870773149814
kxy from the synchronous table  : 35381.870773149814
kxy at 8000 RPM, whirl 2000 RPM : 28817.24716297168
leakage per speed [kg/s]        : [0.02366 0.02366 0.02366]


Speed [RPM],Frequency [RPM],kxx [N/m],kxy [N/m],cxx [N*s/m],cxy [N*s/m]
8000.0,2000.0,-16236.50285,28817.24716,19.62562,51.37123
8000.0,4000.0,-22774.55333,30019.31966,20.38774,52.24055
8000.0,8000.0,-50204.7279,35381.87077,23.68926,55.97903
8000.0,12000.0,-101129.1818,46855.18946,30.33327,63.28407


Lookups follow the axes the table carries. A single value evaluates the synchronous diagonal — this is what every analysis does by default — while a `(frequency, speed)` pair interpolates on the grid. `plot()` draws one curve per speed against the whirl frequency:

In [15]:
w = Q_(8000, "RPM").to("rad/s").m
print("synchronous lookup      :", float(seal_2d.kxy_interpolated(w)))
print(
    "whirl = 0.5 x speed     :",
    float(seal_2d.kxy_interpolated(frequency=0.5 * w, speed=w)),
)

fig = seal_2d.plot(["kxy"], frequency_units="RPM")
fig.show()

synchronous lookup      : 35381.870773149814
whirl = 0.5 x speed     : 30019.319656022155


## 4.1 Modal analysis with a whirl-dependent seal

Put the seal on a rotor and compare three ways of evaluating its coefficients in `run_modal`:

| Call | Coefficient lookup |
|---|---|
| `run_modal(speed)` | synchronous: whirl frequency = rotor speed (the classic result) |
| `run_modal(speed, frequency=f)` | every mode at the fixed whirl frequency `f` |
| `run_modal(speed, matched_whirl=True)` | each mode iterated until the whirl frequency matches its own damped natural frequency |

The converged (or prescribed) whirl frequencies are stored in `ModalResults.whirl_frequency`. Note that the pre-existing `synchronous=True` option is unrelated: it folds the gyroscopic matrix into the mass matrix (Rouch's formulation).

In [16]:
import ross as rs

steel = rs.Material(name="steel", rho=7810, E=211e9, G_s=81.2e9)
shaft = [rs.ShaftElement(L=0.25, idl=0, odl=0.145, material=steel) for _ in range(6)]
disks = [
    rs.DiskElement.from_geometry(n=2, material=steel, width=0.07, i_d=0.145, o_d=0.45),
    rs.DiskElement.from_geometry(n=4, material=steel, width=0.07, i_d=0.145, o_d=0.45),
]
bearings = [
    rs.BearingElement(n=0, kxx=5e7, kyy=6e7, cxx=2e4, cyy=2e4),
    rs.BearingElement(n=6, kxx=5e7, kyy=6e7, cxx=2e4, cyy=2e4),
]
rotor = rs.Rotor(shaft, disks, bearings + [seal_2d])

modal_sync = rotor.run_modal(speed=w, num_modes=8)
modal_fixed = rotor.run_modal(speed=w, num_modes=8, frequency=0.5 * w)
modal_matched = rotor.run_modal(speed=w, num_modes=8, matched_whirl=True)

print(
    f"{'mode':>4} | {'wd sync':>9} | {'wd matched':>10} | {'logdec sync':>11} | {'logdec fixed':>12} | {'logdec matched':>14} | {'whirl freq':>10}"
)
for i in range(4):
    print(
        f"{i:>4} | {modal_sync.wd[i]:>9.2f} | {modal_matched.wd[i]:>10.2f} | "
        f"{modal_sync.log_dec[i]:>11.3f} | {modal_fixed.log_dec[i]:>12.3f} | "
        f"{modal_matched.log_dec[i]:>14.3f} | {modal_matched.whirl_frequency[i]:>10.2f}"
    )

mode |   wd sync | wd matched | logdec sync | logdec fixed | logdec matched | whirl freq
   0 |    388.82 |     388.96 |       0.248 |        0.247 |          0.247 |     388.96
   1 |    407.30 |     407.43 |       0.200 |        0.200 |          0.200 |     407.43
   2 |    968.18 |     968.18 |       1.029 |        1.029 |          1.029 |     968.18
   3 |   1084.28 |    1084.28 |       0.930 |        0.930 |          0.930 |    1084.28


/home/raphaelts/ross-wt-release-3.2.0/ross/rotor_assembly.py:2341: UserWarning: The coefficients of Labyrinth Seal 0 are interpolated from only 4 frequency points. Tabulate at least 5 points per axis spanning the analysis range for reliable interpolation.
  warnings.warn(


The synchronous run evaluates the seal at the running speed while the first modes whirl at a much lower frequency; `matched_whirl` converges each mode onto its own damped natural frequency (compare `whirl_frequency` with `wd`). For this lightly loaded seal the bearings dominate and the log decrements move little; the effect grows with pressure ratio, preswirl and seal length, and can flip the sign of the log decrement of a compressor rotor with several high-pressure seals. The same option is available for the Campbell diagram, where each speed is solved with matched-whirl coefficients:

In [17]:
speed_range = np.linspace(0.5 * w, 1.4 * w, 8)
campbell_matched = rotor.run_campbell(speed_range, frequencies=4, matched_whirl=True)

fig = campbell_matched.plot(harmonics=[1], frequency_units="RPM", speed_units="RPM")
fig.show()

/home/raphaelts/ross-wt-release-3.2.0/ross/rotor_assembly.py:2333: UserWarning: Extrapolating the coefficients of Labyrinth Seal 0 outside its speed axis (523.6 to 1152 rad/s). Be careful when post-processing the results.
  warnings.warn(
/home/raphaelts/ross-wt-release-3.2.0/ross/rotor_assembly.py:2341: UserWarning: The coefficients of Labyrinth Seal 0 are interpolated from only 3 speed points. Tabulate at least 5 points per axis spanning the analysis range for reliable interpolation.
  warnings.warn(
/home/raphaelts/ross-wt-release-3.2.0/ross/rotor_assembly.py:2341: UserWarning: The coefficients of Labyrinth Seal 0 are interpolated from only 4 frequency points. Tabulate at least 5 points per axis spanning the analysis range for reliable interpolation.
  warnings.warn(
/home/raphaelts/ross-wt-release-3.2.0/ross/rotor_assembly.py:2333: UserWarning: Extrapolating the coefficients of Labyrinth Seal 0 outside its frequency axis (209.4 to 1257 rad/s). Be careful when post-processing the re

/home/raphaelts/ross-wt-release-3.2.0/ross/rotor_assembly.py:2333: UserWarning: Extrapolating the coefficients of Labyrinth Seal 0 outside its frequency axis (209.4 to 1257 rad/s). Be careful when post-processing the results.
  warnings.warn(
/home/raphaelts/ross-wt-release-3.2.0/ross/rotor_assembly.py:2341: UserWarning: The coefficients of Labyrinth Seal 0 are interpolated from only 4 frequency points. Tabulate at least 5 points per axis spanning the analysis range for reliable interpolation.
  warnings.warn(
